<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/Restart-From-Hackation_v1.0/mnps_post_getting_started.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNPS Job Equity Post Mini-Hackathon
> A notebook to help you get started  
> DSI DSSG + MNPS Hackathon  
> September 9, 2025  
> Drafted by Wayne Birch - [contact her](wayne.birch@mnps.org) for questions, code update needs, or other questions about the notebook!

This notebook is a restart point based on the work done in the mini Hackathon with Metro Nashville Public Schools (MNPS) and the VU Data Science Institute (VU DSI).

Details from the Hacakation follow:

You aren't constrained to what is in this notebook, and please feel free to use your creativity to deliver the best solution

## **1** | Competition Parameters
* **Outcome and evaluation**: Participants will be evaluated on the performance of their provided solution on the holdout set. Importantly, judges must be able to easily run the submitted code on the new dataset.
* **Objective**: The overall objective is to create a system which best automatically, reproducibly, and reliably categorizes jobs according to the parameters set forth by MNPS. A few suggestions are provided on parameters that you can vary if you're thinking about achievable changes in 2.5 hours


## **2** | Environment Setup
Again, you're completely free to just download this notebook, create a local virtual environment and get to coding in your favorite IDE. We provide this code just as a rapid method to get started, and focus our efforts on implementation through Google Colab.

### **2a** | API Key Setup
#### **2a.1** | Access
The DSI has provided you an API key which can access **some** of the OpenAI models. These include:
* All versions of gpt-4o
* All versions of gpt-4.1
* All versions of o3-mini

Vector store upload, web search, code interpreter, and other functionality outside of the Chat Completions and Messages API is **not** supported. If you really want to use these things, you will have to make a good and cost-supported argument. If you don't feel like arguing, you can also utilize your own OpenAI API key.

#### **2a.2** | API Keys in Google Colab
To use your API key, click on the key icon (looks sort of like 🔑) in the left sidebar.  Under **Name**, add `OPENAI_API_KEY`. Under **Value**, paste your API key. Your API key is a jumble of numbers and letters, maybe even other symbols. Click the slider checkbox to enable **Notebook access** (so your notebook will grab these values without asking you).  

### **2b** | Runtime setup
We're going to install some packages in your environment so that you have access to the code functionality. If you need more packages, install more packages. Install **only** packages you trust.

In [ ]:
!pip install openai

In [ ]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List
import pandas as pd
from google.colab import userdata

# set OpenAI API key environment variable using Google Colab
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# ---- Model selection (single source of truth) ----
# Optional: set an OPENAI_MODEL secret in Colab's 🔑 sidebar (e.g., gpt-4o-2024-11-20).
# If not set, we fall back to a safe default snapshot.
OPENAI_MODEL_FROM_SECRET = userdata.get("OPENAI_MODEL")

# Map friendly names → pinned snapshots (edit as you like)
SNAPSHOTS = {
    "gpt-4o":   "gpt-4o-2024-11-20",
    "gpt-4.1":  "gpt-4.1-2025-04-14",
    "gpt-4.1-mini": "gpt-4.1-mini-2025-04-14",
    "o3-mini":  "o3-mini-2025-01-31",
}

# Default if nothing is set
MODEL_ID = OPENAI_MODEL_FROM_SECRET or "gpt-4o-2024-11-20"
MODEL_ID = SNAPSHOTS.get(MODEL_ID, MODEL_ID)

print("✅ Using model:", MODEL_ID)

## **3** | The Data

The current prompt is a two-step prompt that is successful through the ChatGPT interface. It requires two types of data:
* The data to be classified
* Supporting resources

We need to read all of this in. Let's grab it and use it. The first thing you'll do is just straight up download a zip file of all of this information.

You can download all of the reference files from the link provided, then upload in the sidebar. You'll then unzip the directory using the code below.

Click on the folder icon in the left sidebar (kinda looks like this 🗂️) and you'll see all the files there. We'll read them in.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
base_target_folder = '/content/drive/My Drive/Colab Notebooks/Data Inputs'
!unzip "{base_target_folder}/MNPS_Prompt_Resources.zip" -d /content/

Mounted at /content/drive
Archive:  /content/drive/My Drive/Colab Notebooks/Data Inputs/MNPS_Prompt_Resources.zip
  inflating: /content/Korn_Ferry Lominger 38 Competencies.csv  
  inflating: /content/Competency Extended Descriptions.csv  
  inflating: /content/MNPS KSACs.csv  
  inflating: /content/MNPS Roles.csv  


In [ ]:
import pandas as pd
resources_dir_prefix = '/content/'
roles_lookup = pd.read_csv(resources_dir_prefix+"MNPS Roles.csv")
determinants = pd.read_csv(resources_dir_prefix+"Competency Extended Descriptions.csv", encoding='latin1')
ksac_table = pd.read_csv(resources_dir_prefix+"MNPS KSACs.csv")
korn_ferry = pd.read_csv(resources_dir_prefix+"Korn_Ferry Lominger 38 Competencies.csv", encoding='latin1')

ground_truth_masterfile = pd.read_csv(f"{base_target_folder}/Ground Truth Masterfile.csv", encoding='latin1')
new_sample = pd.read_csv(f"{base_target_folder}/New Sample_08.07.2025.csv", encoding='latin1')

## **4** | The Prompts

What we have here is a direct prompt to get the response that we're looking for. We'll make this happen directly using the OpenAI Chat Completions API. Note that you can use other APIs as you like.

In [ ]:
zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "Job Description Export Specialists.xlsx" file based on similarities in job functions, not job titles.

Process:

- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Compare each job with reference sources using the same attributes. I have attached the reference sources for you.
- Group jobs based on similarities into:
  - Major role groupings (e.g., Specialist, Analyst, Manager)
  - Minor sub-groupings (e.g., Specialist I, II, III, IV) - not to exceed level IV
- Use the MNPS Roles and MNPS KSACs documents to help you determine major role groupings.
- Use the remaining documents to help you clarify subtle differences in role groupings and sub-groupings.
- Use a more qualitative, holistic assessment focused on functional alignment with KSACs rather than a quantitative scoring approach with defined complexity metrics

Output Format:

- Create a table with the following columns:
  - Original Job Title
  - New Job Title
  - Major Role Group
  - Minor Sub-Group
  - Justification for Grouping

- Provide an accompanying narrative explaining the rationale behind the groupings and any notable patterns or insights discovered during the analysis.

Job Title Convention:

- Follow the format: "[Function] [Role] [Level]" (e.g., "Collections Specialist II", "Accounts Payable Specialist III")

Additional Guidelines:

- Ensure all sources used are cited properly.
- Focus on the nature of the work performed rather than just the job titles.
- Consider the complexity of tasks, level of responsibility, and required competencies when determining groupings.
- Provide clear explanations for why each job was classified as it was, referencing specific job attributes and external benchmarks.

"""

Instead of asking for a table output, we will use **structured outputs**. Though this is a common approach for the outputs of LLMs/AI systems, you can learn more about this on [OpenAI's structured output documentation](https://platform.openai.com/docs/guides/structured-outputs?api-mode=responses). Note that you can find this information on almost all LLM/AI platform or package providers.

In [ ]:
from pydantic import BaseModel, Field

class JobClassification(BaseModel):
    """Represents the classification of a job based on its functions."""
    job_title_original: str = Field(..., description="The original job title as provided in the input data using the job title convention specified.")
    new_job_title: str = Field(..., description="The proposed new job title based on the classification using the job title convention specified.")
    major_role_group: str = Field(..., description="The major grouping of the job based on its functional role (e.g., Specialist, Analyst, Manager).")
    minor_sub_group: str = Field(..., description="The minor sub-grouping within the major role group (e.g., Specialist I, II, III, IV).")
    grouping_justification: str = Field(..., description="The justification for placing the job in the specific major and minor groups, referencing job attributes and relevant documents.")

In [ ]:
from typing import List

class JobClassificationTable(BaseModel):
  """The table classification and overall commentary on the groupings provided by the AI system."""
  job_classification_table: List[JobClassification] = Field(..., description="The table of job classifications.")
  narrative_rationale: str = Field(..., description="The narrative commentary on the groupings provided by the AI system.")

Create classifications using OpenAI. Of note here is:
* The **developer** prompt - this is the "system prompt" or "custom instructions" for the model. This determines the overall behavior of the model.
* The **user** prompt - this is what we send to the model like when we're chatting with ChatGPT.

In [ ]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List
from google.colab import userdata

# set OpenAI API key environment variable using Google Colab
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY') <--duplicate already set it in the environment cell

zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "Job Description Export Specialists.xlsx" file based on similarities in job functions, not job titles.

Process:

- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Compare each job with reference sources using the same attributes. I have attached the reference sources for you.
- Group jobs based on similarities into:
  - Major role groupings (e.g., Specialist, Analyst, Manager)
  - Minor sub-groupings (e.g., Specialist I, II, III, IV) - not to exceed level IV
- Use the MNPS Roles and MNPS KSACs documents to help you determine major role groupings.
- Use the remaining documents to help you clarify subtle differences in role groupings and sub-groupings.
- Use a more qualitative, holistic assessment focused on functional alignment with KSACs rather than a quantitative scoring approach with defined complexity metrics

Output Format:

- Create a table with the following columns:
  - Original Job Title
  - New Job Title
  - Major Role Group
  - Minor Sub-Group
  - Justification for Grouping

- Provide an accompanying narrative explaining the rationale behind the groupings and any notable patterns or insights discovered during the analysis.

Job Title Convention:

- Follow the format: "[Function] [Role] [Level]" (e.g., "Collections Specialist II", "Accounts Payable Specialist III")

Additional Guidelines:

- Ensure all sources used are cited properly.
- Focus on the nature of the work performed rather than just the job titles.
- Consider the complexity of tasks, level of responsibility, and required competencies when determining groupings.
- Provide clear explanations for why each job was classified as it was, referencing specific job attributes and external benchmarks.

"""

class JobClassification(BaseModel):
    """Represents the classification of a job based on its functions."""
    job_title_original: str = Field(..., description="The original job title as provided in the input data using the job title convention specified.")
    new_job_title: str = Field(..., description="The proposed new job title based on the classification using the job title convention specified.")
    major_role_group: str = Field(..., description="The major grouping of the job based on its functional role (e.g., Specialist, Analyst, Manager).")
    minor_sub_group: str = Field(..., description="The minor sub-grouping within the major role group (e.g., Specialist I, II, III, IV).")
    grouping_justification: str = Field(..., description="The justification for placing the job in the specific major and minor groups, referencing job attributes and relevant documents.")

class JobClassificationTable(BaseModel):
  """The table classification and overall commentary on the groupings provided by the AI system."""
  job_classification_table: List[JobClassification] = Field(..., description="The table of job classifications.")
  narrative_rationale: str = Field(..., description="The narrative commentary on the groupings provided by the AI system.")


# Create openAI client
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Create messages to send
messages = [
    {"role": "developer", "content": zero_shot_prompt},
    {"role": "user", "content": "Classify the following job description: [Paste Job Description Here]"} # Replace with actual job description
]

# Create openAI client (env var already set in the setup cell)
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

response = client.beta.chat.completions.parse(
    model=MODEL_ID,  # <<— was "gpt-4o"
    messages=messages,
    temperature=1,
    max_tokens=1000,
    response_format=JobClassificationTable
)
# PREVIOUS
# Assuming JobClassification and zero_shot_prompt are defined in the preceding code
# response = client.beta.chat.completions.parse(
    # model="gpt-4o", # Or another available model
    # messages=messages,
    # temperature=1,
    # max_tokens=1000,
    # response_format=JobClassificationTable
# )

print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-CDuo8rqrz9EIEmDaP4GEckbPWrnSt",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\"job_classification_table\":[],\"narrative_rationale\":\"To classify a job description provided as input, the system would first compare the essential functions, knowledge, skills, abilities, education, work experience, and certifications as outlined in the job description against a set of reference sources and organizational standards.\\n\\nThe system uses a qualitative analysis to assess the level of responsibility, complexity, and required competencies to determine appropriate groupings. This ensures that each job is aligned with major role groupings such as Specialist, Analyst, Manager, and further delineated into minor sub-groups like levels I, II, III, or IV, if applicable.\\n\\nFor example, if a job description involves routine data analysis tasks that require moderate technical skills and exper

In [ ]:
#look at response
response.choices[0].message.parsed

JobClassificationTable(job_classification_table=[], narrative_rationale='To classify a job description provided as input, the system would first compare the essential functions, knowledge, skills, abilities, education, work experience, and certifications as outlined in the job description against a set of reference sources and organizational standards.\n\nThe system uses a qualitative analysis to assess the level of responsibility, complexity, and required competencies to determine appropriate groupings. This ensures that each job is aligned with major role groupings such as Specialist, Analyst, Manager, and further delineated into minor sub-groups like levels I, II, III, or IV, if applicable.\n\nFor example, if a job description involves routine data analysis tasks that require moderate technical skills and experience, it might be categorized as a "Data Analyst II" under the Analyst major role group, reflecting both the function (Data Analysis) and the level of expertise or responsibi

We can make this into a table using pandas!

In [ ]:
response_dict = dict(*response.choices[0].message.parsed.job_classification_table)
response_dict

{}

In [ ]:
# see outputs
pd.DataFrame(response_dict, index=[0])

""
0


In [ ]:
import os
import datetime
import shutil
import ipykernel

# Get the notebook name
try:
    # This method works in Colab
    notebook_path = ipykernel.get_connection_file()
    notebook_name = os.path.basename(notebook_path).split('.')[0]
except:
    # Fallback for other environments
    notebook_name = 'Colab_Notebook_Run'

# Define the destination directory in Google Drive
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
destination_dir = f'/content/drive/My Drive/Colab Notebooks/Run Results/{timestamp}_{notebook_name}'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# List of files to copy (modify this list as needed)
files_to_copy = [
    '/content/Korn_Ferry Lominger 38 Competencies.csv',
    '/content/Competency Extended Descriptions.csv',
    '/content/MNPS KSACs.csv',
    '/content/MNPS Roles.csv',
    f'{base_target_folder}/Ground Truth Masterfile.csv', # Copying from the original location
    f'{base_target_folder}/New Sample_08.07.2025.csv', # Copying from the original location
    # Add any other files you want to copy from the run, e.g., output files
    # '/content/your_output_file.csv'
]

# Copy the files
for file_path in files_to_copy:
    try:
        shutil.copy(file_path, destination_dir)
        print(f"Copied: {file_path} to {destination_dir}")
    except FileNotFoundError:
        print(f"File not found: {file_path}")
    except Exception as e:
        print(f"Error copying {file_path}: {e}")

print("File copying complete.")

Copied: /content/Korn_Ferry Lominger 38 Competencies.csv to /content/drive/My Drive/Colab Notebooks/Run Results/20250909_154458_kernel-05d9c2a8-87f7-44de-9515-f37bc0ab202e
Copied: /content/Competency Extended Descriptions.csv to /content/drive/My Drive/Colab Notebooks/Run Results/20250909_154458_kernel-05d9c2a8-87f7-44de-9515-f37bc0ab202e
Copied: /content/MNPS KSACs.csv to /content/drive/My Drive/Colab Notebooks/Run Results/20250909_154458_kernel-05d9c2a8-87f7-44de-9515-f37bc0ab202e
Copied: /content/MNPS Roles.csv to /content/drive/My Drive/Colab Notebooks/Run Results/20250909_154458_kernel-05d9c2a8-87f7-44de-9515-f37bc0ab202e
Copied: /content/drive/My Drive/Colab Notebooks/Data Inputs/Ground Truth Masterfile.csv to /content/drive/My Drive/Colab Notebooks/Run Results/20250909_154458_kernel-05d9c2a8-87f7-44de-9515-f37bc0ab202e
Copied: /content/drive/My Drive/Colab Notebooks/Data Inputs/New Sample_08.07.2025.csv to /content/drive/My Drive/Colab Notebooks/Run Results/20250909_154458_kerne